# PatchCore 2차 실행 (광택 시편 제외 / 2배 확대)

**런타임 → 런타임 유형 변경 → T4 GPU** 선택 후 위에서부터 실행.

- 실행 A: N03·N06(광택 반사)을 학습에서 빼고 테스트로 → `results_A`
- 실행 B: A + 입력 2배 확대 → `results_B` (10~20분)
- 1차 결과(`results`)는 그대로 두고 비교에 씀

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/specimen_kit'
!nvidia-smi -L
!pip -q install pillow-heif
!grep -c "exclude-train" "$BASE/patchcore_cv.py"   # 1 이상이어야 새 코드

In [ ]:
# 실행 A
import os
os.chdir(BASE)
!python patchcore_cv.py --kit . --raw raw --out results_A --exclude-train N03,N06 2>&1 | grep -v -i warning | grep -v '^features'

In [ ]:
# 실행 B (2배 확대)
import os
os.chdir(BASE)
!python patchcore_cv.py --kit . --raw raw --out results_B --exclude-train N03,N06 --scale 2 2>&1 | grep -v -i warning | grep -v '^features'

In [ ]:
# 1차 / A / B 비교표
import json, os
def pct(x): return f"{x[0]}/{x[1]} ({100*x[0]/x[1]:.0f}%)" if x[1] else '-'
runs = [(n, f'{BASE}/{d}/summary.json') for n, d in (('1차','results'),('A','results_A'),('B','results_B'))]
runs = [(n, json.load(open(p))) for n, p in runs if os.path.exists(p)]
for tn, name in (('strict','엄격'), ('p999','완화')):
    print(f'\n===== {name} 기준 =====')
    print('실행 | 정상A 과검 | 정상B 과검 | 구멍 밖 과검 | 구멍 검출 | 1-1.5 | 1.5-2 | 2-2.5 | 2.5-3 | 3+')
    for n, S in runs:
        a = S['aggregate'][tn]; hs = a['holes_by_size']
        print(n, '|', pct(a['false_alarm_A']), '|', pct(a['false_alarm_B']), '|', pct(a['pinhole_background_alarm_imgs']), '|', pct(a['holes_detected']), '|', ' | '.join(pct(hs[k]) for k in hs))
    for n, S in runs:
        a = S['aggregate'][tn]
        print(n, '이물·줄무늬:', a['defect_imgs_over_thr'], '| 광택 제외 시편:', a.get('excluded_A_over', '-'))
for n, S in runs:
    print(n, 'AUROC', round(S['aggregate']['image_auroc_mean'], 3), '| 엄격 기준값', [round(S['folds'][f]['strict']['threshold'], 2) for f in S['folds']])

In [ ]:
# 결과 압축 → specimen_kit/results_v2.zip 을 채팅에 올리기
!cd "$BASE" && rm -f results_v2.zip && zip -qr results_v2.zip results_A results_B -x '*.npy' && ls -la results_v2.zip

In [ ]:
# 2단계 분류용 점수 지도 내보내기 (float16 압축)
import numpy as np, glob, os
os.chdir(BASE)
for f in ['A_fold1', 'A_fold2', 'A_fold3']:
    files = glob.glob(f'results_B/{f}/*_score.npy')
    d = {os.path.basename(p).replace('_score.npy', ''): np.load(p).astype(np.float16) for p in files}
    np.savez_compressed(f'scores_B_{f}.npz', **d)
    print(f, len(d), round(os.path.getsize(f'scores_B_{f}.npz') / 1e6, 1), 'MB')